In [1]:
from pathlib import Path
import base64
import re
import xml.etree.ElementTree as ET

import pandas as pd
import plotly.graph_objects as go
from plotly.colors import sample_colorscale

In [ ]:
DATA_DIR = Path("../data")
ASSET_DIR = Path("../assets")

PARQUET_PATH = DATA_DIR / "red_par_sample.parquet"
SVG_PATH = ASSET_DIR / "greenhouse.svg"

GREENHOUSE_TZ = "Europe/Berlin"

USE_LATEST_DATE_IN_DATA = True # for development
# USE_LATEST_DATE_IN_DATA = False # FOR PRODUCTION

sensor_to_device = {
    "s_01": "s2100:s2100-01-par",
    "s_02": "s2100:s2100-02-par",
    "s_10": "s2100:s2100-10-par",
    "s_11": "s2100:s2100-11-par",
    "s_12": "s2100:s2100-12-par",
    "s_13": "s2100:s2100-13-par",
    "s_14": "s2100:s2100-14-par",
}

PAR_COLORSCALE = [
    [0.00, "#fff7bc"],
    [0.25, "#fee391"],
    [0.50, "#fec44f"],
    [0.75, "#fe9929"],
    [1.00, "#cc4c02"],
]

In [3]:
df_par = pd.read_parquet(PARQUET_PATH)

df_par["time"] = pd.to_datetime(df_par["time"], utc=True)
df_par["value"] = pd.to_numeric(df_par["value"], errors="coerce")

df_par = df_par.dropna(subset=["time", "device", "value"]).copy()

df_par.tail()

,device,sensor,time,value
68548,s2100:s2100-14-par,par,2026-04-28 14:36:17+00:00,94.0
68549,s2100:s2100-01-par,par,2026-04-28 14:39:32+00:00,294.0
68550,s2100:s2100-10-par,par,2026-04-28 14:40:28+00:00,709.0
68551,s2100:s2100-13-par,par,2026-04-28 14:40:38+00:00,9.0
68552,s2100:s2100-11-par,par,2026-04-28 14:41:50+00:00,649.0


In [4]:
(
    df_par
    .groupby("device")
    .agg(
        n=("value", "size"),
        min_time=("time", "min"),
        max_time=("time", "max"),
        min_value=("value", "min"),
        median_value=("value", "median"),
        max_value=("value", "max"),
    )
    .sort_index()
)

,n,min_time,max_time,min_value,median_value,max_value
device,,,,,,
s2100:s2100-01-par,27761,2025-10-09 06:55:03+00:00,2026-04-28 14:39:32+00:00,0.0,0.0,2802.0
s2100:s2100-02-par,27770,2025-10-09 07:01:07+00:00,2026-04-28 14:35:23+00:00,0.0,158.0,2266.0
s2100:s2100-10-par,3019,2026-04-07 12:56:37+00:00,2026-04-28 14:40:28+00:00,0.0,48.0,1744.0
s2100:s2100-11-par,3007,2026-04-07 12:53:11+00:00,2026-04-28 14:41:50+00:00,0.0,44.0,1757.0
s2100:s2100-12-par,959,2026-04-07 13:23:41+00:00,2026-04-15 17:19:44+00:00,0.0,46.0,1548.0
s2100:s2100-13-par,3022,2026-04-07 12:57:25+00:00,2026-04-28 14:40:38+00:00,0.0,4.0,1457.0
s2100:s2100-14-par,3015,2026-04-07 12:57:45+00:00,2026-04-28 14:36:17+00:00,0.0,12.0,1743.0


In [5]:
def svg_to_data_uri(path: Path) -> str:
    encoded = base64.b64encode(path.read_bytes()).decode("utf-8")
    return f"data:image/svg+xml;base64,{encoded}"


def parse_viewbox(svg_path: Path):
    root = ET.parse(svg_path).getroot()
    viewbox = root.attrib.get("viewBox")

    if viewbox:
        x, y, w, h = map(float, viewbox.split())
        return x, y, w, h

    width = float(re.sub(r"[^0-9.]", "", root.attrib["width"]))
    height = float(re.sub(r"[^0-9.]", "", root.attrib["height"]))
    return 0.0, 0.0, width, height


def parse_svg_rects(svg_path: Path):
    ns = {"svg": "http://www.w3.org/2000/svg"}
    root = ET.parse(svg_path).getroot()

    rects = {}

    for rect in root.findall(".//svg:rect", ns):
        rect_id = rect.attrib.get("id")
        if not rect_id:
            continue

        rects[rect_id] = {
            "x": float(rect.attrib.get("x", 0)),
            "y": float(rect.attrib.get("y", 0)),
            "width": float(rect.attrib.get("width", 0)),
            "height": float(rect.attrib.get("height", 0)),
            "rx": rect.attrib.get("rx"),
        }

    return rects


def svg_rect_to_plotly_rect(rect, canvas_h):
    x0 = rect["x"]
    x1 = rect["x"] + rect["width"]

    # SVG y=0 top, Plotly y=0 bottom
    y0 = canvas_h - (rect["y"] + rect["height"])
    y1 = canvas_h - rect["y"]

    return x0, x1, y0, y1

In [6]:
_, _, canvas_w, canvas_h = parse_viewbox(SVG_PATH)
rects = parse_svg_rects(SVG_PATH)

sensor_boxes = {
    k: v for k, v in rects.items()
    if k.startswith("s_") and not k.endswith("_bg")
}

sensor_bands = {
    k.replace("_bg", ""): v for k, v in rects.items()
    if k.startswith("s_") and k.endswith("_bg")
}

canvas_w, canvas_h, sensor_boxes.keys(), sensor_bands.keys()

(670.0,
 609.0,
 dict_keys(['s_01', 's_02', 's_10', 's_14', 's_13', 's_12', 's_11']),
 dict_keys(['s_01', 's_02', 's_12', 's_10', 's_11', 's_14', 's_13']))

In [7]:
def filter_day(df, use_latest_date_in_data=True):
    df_local = df.copy()
    df_local["time_local"] = df_local["time"].dt.tz_convert(GREENHOUSE_TZ)

    if use_latest_date_in_data:
        target_day = df_local["time_local"].max().normalize()
    else:
        target_day = pd.Timestamp.now(tz=GREENHOUSE_TZ).normalize()

    next_day = target_day + pd.Timedelta(days=1)

    mask = (
        (df_local["time_local"] >= target_day) &
        (df_local["time_local"] < next_day)
    )

    return df_local.loc[mask].drop(columns=["time_local"]).copy(), target_day


df_day, target_day = filter_day(df_par, USE_LATEST_DATE_IN_DATA)

target_day, df_day.shape

(Timestamp('2026-04-28 00:00:00+0200', tz='Europe/Berlin'), (601, 4))

In [8]:
def compute_dli(sensor_df):
    d = sensor_df.sort_values("time").copy()

    if len(d) < 2:
        return None

    d["next_time"] = d["time"].shift(-1)
    d["next_value"] = d["value"].shift(-1)

    d["dt_seconds"] = (d["next_time"] - d["time"]).dt.total_seconds()

    # huge gaps in data
    d["dt_seconds"] = d["dt_seconds"].clip(lower=0, upper=900)

    # trapezoid integration
    d["avg_value"] = (d["value"] + d["next_value"]) / 2

    d = d.dropna(subset=["dt_seconds", "avg_value"])

    return float((d["avg_value"] * d["dt_seconds"]).sum() / 1_000_000)


def compute_sensor_metrics(df_day):
    rows = []

    for sensor_id, device in sensor_to_device.items():
        d = (
            df_day[df_day["device"] == device]
            .sort_values("time")
            .copy()
        )

        if d.empty:
            rows.append({
                "sensor_id": sensor_id,
                "device": device,
                "latest_par": None,
                "latest_time": None,
                "dli_today": None,
                "n": 0,
            })
            continue

        latest = d.iloc[-1]

        rows.append({
            "sensor_id": sensor_id,
            "device": device,
            "latest_par": float(latest["value"]),
            "latest_time": latest["time"],
            "dli_today": compute_dli(d),
            "n": len(d),
        })

    return pd.DataFrame(rows)


metrics = compute_sensor_metrics(df_day)
metrics

,sensor_id,device,latest_par,latest_time,dli_today,n
0,s_01,s2100:s2100-01-par,294.0,2026-04-28 14:39:32+00:00,49.061667,100
1,s_02,s2100:s2100-02-par,224.0,2026-04-28 14:35:23+00:00,29.624631,98
2,s_10,s2100:s2100-10-par,709.0,2026-04-28 14:40:28+00:00,23.031046,101
3,s_11,s2100:s2100-11-par,649.0,2026-04-28 14:41:50+00:00,16.891656,101
4,s_12,s2100:s2100-12-par,NaN,NaT,NaN,0
5,s_13,s2100:s2100-13-par,9.0,2026-04-28 14:40:38+00:00,0.653709,101
6,s_14,s2100:s2100-14-par,94.0,2026-04-28 14:36:17+00:00,2.938250,100


In [9]:
def value_to_color(value, vmin, vmax, colorscale=PAR_COLORSCALE, alpha=None):
    if value is None or pd.isna(value):
        return "rgba(180,180,180,0.45)"

    if vmax <= vmin:
        t = 0.5
    else:
        t = (value - vmin) / (vmax - vmin)
        t = max(0, min(1, t))

    color = sample_colorscale(colorscale, [t])[0]

    if alpha is None:
        return color

    # plotly returns rgb(r,g,b)
    nums = re.findall(r"\d+", color)
    r, g, b = nums[:3]
    return f"rgba({r},{g},{b},{alpha})"

In [12]:
def make_par_greenhouse_plot(metrics):
    fig = go.Figure()

    latest_values = metrics["latest_par"].dropna()
    dli_values = metrics["dli_today"].dropna()

    latest_min = float(latest_values.min()) if not latest_values.empty else 0
    latest_max = float(latest_values.max()) if not latest_values.empty else 1

    dli_min = float(dli_values.min()) if not dli_values.empty else 0
    dli_max = float(dli_values.max()) if not dli_values.empty else 1

    # background SVG
    fig.add_layout_image(
        dict(
            source=svg_to_data_uri(SVG_PATH),
            xref="x",
            yref="y",
            x=0,
            y=canvas_h,
            sizex=canvas_w,
            sizey=canvas_h,
            sizing="stretch",
            opacity=1,
            layer="below",
        )
    )

    # draw DLI bands first
    for _, row in metrics.iterrows():
        sid = row["sensor_id"]

        if sid not in sensor_bands:
            continue

        x0, x1, y0, y1 = svg_rect_to_plotly_rect(sensor_bands[sid], canvas_h)

        band_color = value_to_color(
            row["dli_today"],
            dli_min,
            dli_max,
            colorscale=PAR_COLORSCALE,
            alpha=0.45,
        )

        fig.add_shape(
            type="rect",
            x0=x0,
            x1=x1,
            y0=y0,
            y1=y1,
            line=dict(width=0),
            fillcolor=band_color,
            layer="below",
        )

    # draw sensor boxes + labels
    for _, row in metrics.iterrows():
        sid = row["sensor_id"]

        if sid not in sensor_boxes:
            continue

        x0, x1, y0, y1 = svg_rect_to_plotly_rect(sensor_boxes[sid], canvas_h)

        sensor_color = value_to_color(
            row["latest_par"],
            latest_min,
            latest_max,
            colorscale=PAR_COLORSCALE,
            alpha=0.95,
        )

        fig.add_shape(
            type="rect",
            x0=x0,
            x1=x1,
            y0=y0,
            y1=y1,
            line=dict(color="black", width=1.5),
            fillcolor=sensor_color,
            layer="above",
        )

        label = "—" if row["latest_par"] is None or pd.isna(row["latest_par"]) else f"{row['latest_par']:.0f}"

        fig.add_annotation(
            x=(x0 + x1) / 2,
            y=(y0 + y1) / 2,
            text=label,
            showarrow=False,
            font=dict(size=11, color="black"),
            xanchor="center",
            yanchor="middle",
        )

    # colorbar for latest PAR
    fig.add_trace(
        go.Scatter(
            x=[None],
            y=[None],
            mode="markers",
            marker=dict(
                color=[latest_min, latest_max],
                colorscale=PAR_COLORSCALE,
                cmin=latest_min,
                cmax=latest_max,
                showscale=True,
                colorbar=dict(title="PAR"),
            ),
            showlegend=False,
            hoverinfo="skip",
        )
    )

    fig.update_layout(
        title=f"PAR profile — {target_day.date()}",
        width=850,
        height=760,
        margin=dict(l=20, r=20, t=60, b=20),
        plot_bgcolor="white",
    )

    fig.update_xaxes(
        range=[0, canvas_w],
        visible=False,
        fixedrange=True,
    )

    fig.update_yaxes(
        range=[0, canvas_h],
        visible=False,
        scaleanchor="x",
        scaleratio=1,
        fixedrange=True,
    )

    return fig

In [ ]:
fig = make_par_greenhouse_plot(metrics)
fig.show()

In [ ]:
# # save html
# OUT_DIR = Path("../outputs")
# OUT_DIR.mkdir(parents=True, exist_ok=True)

# out_path = OUT_DIR / "par_greenhouse_profile.html"
# fig.write_html(out_path)

# out_path